# SS-bSSFP and FLASH spoiler validation

This notebook measures the end-volume spoiler in a saved Bloch Simulator project. It reports phase cycles across one **phantom voxel**, compares subvoxel grids, checks RF/phantom frequency matching, reproduces the FLASH through-slice example, and can optionally reconstruct every metabolite separately.

In [ ]:
from pathlib import Path
import runpy
import pandas as pd
from IPython.display import display, Image

root = Path.cwd().resolve()
if not (root / 'pyproject.toml').exists():
    root = root.parent
helpers = runpy.run_path(str(root / 'examples' / 'validate_ss_bssfp_spoiling.py'))
project_path = root / 'exports' / 'debug' / 'bssfp_no_pyr_fa.blochproj'
project_path

## Fast check

The continuous value is the exact rectangular-voxel result. The grid values show what regular 2×2×2 through 8×8×8 subvoxel sampling calculates.

In [ ]:
analysis = helpers['analyze_project'](project_path)
summary = pd.Series({
    'phantom shape': analysis['phantom_shape'],
    'voxel size XYZ [mm]': tuple(1000 * value for value in analysis['phantom_voxel_size_m_xyz']),
    'spoiler cycles/voxel XYZ': analysis['spoiler_cycles_per_voxel_xyz'],
    'continuous retained signal [%]': 100 * analysis['continuous_retained_coherence'],
})
display(summary.to_frame('value'))
display(pd.DataFrame(analysis['grid_results']).assign(
    retained_percent=lambda table: 100 * table['retained_coherence']
))
display(pd.DataFrame(analysis['frequency_comparison']))

## Grid convergence and aliasing

One cycle per voxel is the green recommendation. Peaks at larger integer cycle counts show where a regular grid can accidentally sample the same phase repeatedly.

In [ ]:
plot_path = root / 'exports' / 'debug' / 'ss_bssfp_spoiler_validation.png'
helpers['plot_spoiler_response'](
    plot_path, analysis['spoiler_cycles_per_voxel_xyz']
)
display(Image(filename=str(plot_path)))

## FLASH example from the screenshots

Four spoiler cycles across a 3 mm slice are only two thirds of a cycle across a 0.5 mm phantom voxel. The physical-gradient simulation therefore keeps substantial transverse coherence, whereas the Ideal Crusher sets it to zero by definition.

In [ ]:
flash = helpers['flash_through_slice_spoiler_report'](
    cycles_per_slice=4.0,
    slice_thickness_m=3e-3,
    phantom_voxel_size_m=0.5e-3,
    subvoxel_count=4,
)
pd.Series({
    'cycles per phantom voxel': flash['cycles_per_phantom_voxel'],
    'continuous retained signal [%]': 100 * flash['continuous_retained_coherence'],
    '4-point retained signal [%]': 100 * flash['grid_retained_coherence'],
    'cycles/slice for one cycle/voxel': flash['cycles_per_slice_for_one_cycle_per_voxel'],
}).to_frame('value')

## Optional metabolite-resolved reconstruction

Set `RUN_SPECIES = True` to simulate every spectral component separately with the physical gradient and with the ideal crusher. This is slower, but directly reveals which metabolite appears in each target volume. The quick setting uses one spin per voxel; increase `spin_sampling` in the function call for a convergence study.

In [ ]:
RUN_SPECIES = False
if RUN_SPECIES:
    component_rows = helpers['component_image_norms'](
        project_path, timestep_s=20e-6, spin_sampling=(1, 1, 1)
    )
    component_table = pd.DataFrame(component_rows)
    display(component_table)
    display(component_table.pivot_table(
        index=['component', 'volume'],
        columns='spoiler_mode',
        values='image_l2_norm',
    ))

## What a corrected GUI run should show

For the supplied phantom, enable **Use selected phantom peak frequencies**, set the voxel-referenced spoiler to **1 cycle/voxel**, keep the legacy FOV spoiler at zero, and select physical gradient spoiling with subvoxel spins. The GUI's Spoiler check should report approximately zero retained coherent signal without an aliasing warning.